# Extended Data Figure 5 — Venn diagram of L2G gene prioritisation reasons

Four-way Venn diagram showing the overlap between four direct reasons a (CS, gene) pair
is prioritised by the L2G model:

- **eQTL colocalisation**
- **pQTL colocalisation**
- **VEP** — protein-altering variant
- **TSS** — nearest transcription start site

Numbers indicate the count of CS–gene prioritisations in each overlap region.

**Data:** `data/intermediate_files/list_of_prioritised_genes_per_CS.parquet`


## Load data


In [ ]:
import pandas as pd

path_intermediate = path_to_intermediate_data_folder + ""

# Load prioritised CS-gene pairs and restrict to qualifying disease + measurement CSs
l2g = pd.read_parquet(path_intermediate + "prioritised_genes_per_cs")

qd_cs = pd.read_parquet(path_intermediate + "qualifying_credible_sets", columns=["studyLocusId"])
qm_cs = pd.read_parquet(path_intermediate + "qualifying_measurement_credible_sets", columns=["studyLocusId"])
qualifying_cs = pd.concat([qd_cs, qm_cs]).drop_duplicates()

l2g = l2g.merge(qualifying_cs, on="studyLocusId", how="inner")
print(f"L2G prioritised CS-gene pairs (disease + measurement CSs): {len(l2g):,}")
print(l2g[["eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"]].sum().to_string())

In [ ]:
import matplotlib.pyplot as plt
import venn

ids = l2g["studyLocusId"] + "_" + l2g["geneId"]

sets = {
    "eQTL": set(ids[l2g["eQTL_coloc"] == 1]),
    "pQTL": set(ids[l2g["pQTL_coloc"] == 1]),
    "PAV": set(ids[l2g["VEP"] == 1]),
    "TSS": set(ids[l2g["distanceTSS"] == 1]),
}

fig, ax = plt.subplots(figsize=(8, 8))
venn.venn(sets, ax=ax, fontsize=9, legend_loc="center", cmap=["#8B7FC7", "#6AAC5B", "#D46A6A", "#5EC4CB"])

# Reformat legend to 2 columns
ax.legend(
    handles=ax.get_legend().legend_handles,
    labels=["eQTL", "pQTL", "PAV", "TSS"],
    loc="upper center",
    ncols=2,
    fontsize=10,
    frameon=True,
)

ax.set_title(
    "Venn diagram of four direct reasons\nwhy genes could be prioritised by the L2G framework",
    fontsize=11,
    fontweight="bold",
    pad=12,
)

fig.tight_layout()
fig.savefig(f"{figure_dir}/extended_figure_5.pdf", dpi=300, bbox_inches="tight")
plt.show()